In [3]:
import pandas as pd
import numpy as np
import re

BASE_PATH       = r'C:\Users\gehan\Downloads\thesis\extracted tables'

ENCODED_PATH    = BASE_PATH + r'\combined_df_encoded.parquet'       # base file
ARRIVAL_XLSX    = BASE_PATH + r'\Arrival delay - 20250101-20260531 (1).xlsx'
PNR_CSV         = BASE_PATH + r'\checkin_records_20250101-20260522.csv'
FRIENDS_PATH    = BASE_PATH + r'\survey_and_operational_data.parquet'  # friend's file for wifi PM
OUTPUT_PATH     = BASE_PATH + r'\last update data\combined_df_final.parquet'

print('Paths set.')

Paths set.


In [2]:
BASE_PATH       = r'C:\Users\gehan\Downloads\thesis\extracted tables'
OUTPUT_PATH     = BASE_PATH + r'\last update data\combined_df_final.parquet'

In [3]:
df = pd.read_parquet(ENCODED_PATH)
print(f'Base df: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'arrival_delay_minute: mean={df["arrival_delay_minute"].mean():.1f}, min={df["arrival_delay_minute"].min()}, max={df["arrival_delay_minute"].max()}')

Base df: 178,384 rows x 211 columns
arrival_delay_minute: mean=12.2, min=-106.0, max=739.0


In [4]:
all_null = [(c, df[c].isnull().mean(), df[c].nunique()) for c in df.columns]
all_null_df = pd.DataFrame(all_null, columns=['column', 'null_pct', 'unique'])
all_null_df = all_null_df.sort_values('null_pct', ascending=False)

print('Columns with >80% null:')
print(all_null_df[all_null_df['null_pct'] > 0.8].to_string(index=False))
print(f'\nColumns with 100% null:')
print(all_null_df[all_null_df['null_pct'] == 1.0].to_string(index=False))

Columns with >80% null:
                                                                                                   column  null_pct  unique
                                                     question_on_board_the_train1_the_atmosphere_on_board  1.000000       0
                                          question_on_board_the_train1_the_cleanliness_on_board_the_train  1.000000       0
                                               question_on_board_the_train1_the_level_of_comfort_on_board  1.000000       0
                                                          question_on_board_the_train2_the_wi_fi_on_board  1.000000       0
                                                                                       pm_days_since_wifi  1.000000       0
                                                                                    restriction_days_WiFi  0.998912     104
                                                                                          question_q17_op  0

In [5]:
drop_cols = [
    # 100% null survey columns
    'question_on_board_the_train1_the_atmosphere_on_board',
    'question_on_board_the_train1_the_level_of_comfort_on_board',
    'question_on_board_the_train1_the_cleanliness_on_board_the_train',
    'question_on_board_the_train2_the_wi_fi_on_board',
    # Lounge — Premier only, not operationally linkable
    'question_lounge_overall_f_b_your_overall_experience_in_the_premier_lounge',
    'question_lounge_overall_f_b_the_beverage_offering_in_the_lounge',
    'question_lounge_overall_f_b_available_space_seating',
    'question_lounge_overall_f_b_lounge_staff',
    'question_lounge_overall_f_b_cocktail_bar',
    'question_lounge_overall_f_b_newspapers_magazines',
    'question_lounge_overall_f_b_your_welcome_arrival_at_the_lounge',
    'question_lounge_overall_f_b_atmosphere_ambience_in_the_lounge',
    'question_lounge_overall_f_b_furniture_and_d_cor_in_the_lounge',
    'question_lounge_overall_f_b_information_regarding_your_train_departure',
    'question_lounge_overall_f_b_cleanliness_of_the_lounge',
    'question_lounge_overall_f_b_the_food_offering_in_the_lounge',
    'question_lounge_improvement',
    'question_lounge_usage',
    # Café — 98% null, not operationally linkable
    'question_eurostar_cafe_your_overall_experience_at_the_eurostar_caf',
    'question_eurostar_cafe_the_service_you_received_from_staff',
    'question_eurostar_cafe_the_quality_of_the_food',
    'question_eurostar_cafe_the_quality_of_the_drinks',
    'question_eurostar_cafe_the_range_of_food_on_offer',
    'question_eurostar_cafe_the_range_of_drinks_on_offer',
    'question_eurostar_cafe_the_prices_value_for_money_of_food_and_drinks',
    'question_eurostar_cafe_the_product_availabilty',
    'question_purchase_cafe',
    'question_visit_cafe',
    'question_usage_hv',
    # Cleaning impact block
    'question_impact_of_cleaning_p_this_cleaning_process_is_important_to_ensure_your_comfort_on_the_train',
    'question_impact_of_cleaning_p_this_cleaning_process_helps_to_reinforce_your_satisfaction_with_eurostar',
    'question_impact_of_cleaning_p_this_cleaning_process_encourages_you_to_choose_eurostar_again_in_the_future',
    # Onboard catering overall (duplicate of individual items)
    'question_onboard_catering_ove',
]

drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)
print(f'Dropped {len(drop_cols)} columns — shape now {df.shape}')

Dropped 33 columns — shape now (178384, 178)


In [6]:
arrival_raw = pd.read_excel(ARRIVAL_XLSX)


arrival_raw['Departure Date'] = pd.to_datetime(arrival_raw['Departure Date'], errors='coerce')
arrival_raw = arrival_raw.dropna(subset=['Departure Date', 'Movement Number'])

arrival_raw['service_id'] = arrival_raw['Movement Number'].astype(str) + '_' + arrival_raw['Departure Date'].dt.strftime('%Y%m%d')

cause_cols = arrival_raw[[
    'service_id',
    'Cause Category',
    'Cause Domain',
    'Cause Responsibility',
    'Actor Name'
]].copy().rename(columns={
    'Cause Category':       'delay_cause_category',
    'Cause Domain':         'delay_cause_domain',
    'Cause Responsibility': 'delay_cause_responsibility',
    'Actor Name':           'delay_actor'
})


cause_cols = cause_cols.drop_duplicates(subset='service_id', keep='first')
print(f'Cause rows after dedup: {len(cause_cols):,}')
print(cause_cols['delay_cause_responsibility'].value_counts(dropna=False))

Cause rows after dedup: 46,211
delay_cause_responsibility
External    26738
NaN         11488
Internal     7619
n.a.          366
Name: count, dtype: int64


In [7]:

df = df.merge(cause_cols, on='service_id', how='left')
print(f'Shape after cause join: {df.shape}')
print(f'Cause nulls: {df["delay_cause_responsibility"].isnull().mean():.1%}')

Shape after cause join: (178384, 182)
Cause nulls: 38.6%


In [8]:

print([c for c in df.columns if 'wifi' in c.lower()])

['question_overall_satisfaction_wifi_onboard_the_train', 'restriction_open_WiFi', 'restriction_days_WiFi', 'pm_days_since_wifi', 'pm_has_prior_wifi', 'cm_open_wifi']


In [9]:

df_friends = pd.read_parquet(FRIENDS_PATH)
print(f'Friends file: {df_friends.shape}')
print([c for c in df_friends.columns if 'wifi' in c.lower()])

Friends file: (178833, 211)
['question_overall_satisfaction_wifi_onboard_the_train', 'restriction_open_WiFi', 'restriction_days_WiFi', 'pm_days_since_wifi', 'pm_has_prior_wifi', 'cm_open_wifi']


In [10]:

wifi_pm = df_friends[['service_id', 'pm_days_since_wifi', 'pm_has_prior_wifi']].drop_duplicates(subset='service_id', keep='first')
print(f'wifi_pm after dedup: {len(wifi_pm):,} rows')

df = df.drop(columns=[c for c in ['pm_days_since_wifi', 'pm_has_prior_wifi'] if c in df.columns])


df = df.merge(wifi_pm, on='service_id', how='left')


if 'pm_has_prior_wifi_x' in df.columns:
    df = df.drop(columns=['pm_days_since_wifi_x', 'pm_has_prior_wifi_x'])
    df = df.rename(columns={'pm_days_since_wifi_y': 'pm_days_since_wifi', 'pm_has_prior_wifi_y': 'pm_has_prior_wifi'})

print(f'Total rows: {len(df):,} (expected 178,384)')
print(df['pm_has_prior_wifi'].value_counts(dropna=False))
print(f'pm_days_since_wifi null %: {df["pm_days_since_wifi"].isnull().mean():.1%}')

wifi_pm after dedup: 45,864 rows
Total rows: 178,384 (expected 178,384)
pm_has_prior_wifi
0.0    117170
1.0     59684
NaN      1530
Name: count, dtype: int64
pm_days_since_wifi null %: 66.5%


## PNR Data Join

In [11]:
df_pnr = pd.read_csv(PNR_CSV)
print(f'PNR raw: {df_pnr.shape}')

df_pnr = df_pnr.drop_duplicates(subset='tcn')

df_pnr = df_pnr[df_pnr['train_departure_date'].astype(str).str.len() == 10]


df_pnr['departure_date_clean'] = pd.to_datetime(df_pnr['train_departure_date'], format='%Y-%m-%d').dt.strftime('%Y%m%d')
df_pnr['service_id_pnr'] = df_pnr['train_number'].astype(str) + '_' + df_pnr['departure_date_clean']


pnr_services = set(df_pnr['service_id_pnr'].unique())
df_services  = set(df['service_id'].unique())
overlap_pnr  = pnr_services & df_services
print(f'PNR overlap: {len(overlap_pnr):,} of {len(df_services):,} services ({len(overlap_pnr)/len(df_services):.1%})')

PNR raw: (7149730, 15)
PNR overlap: 20,572 of 46,322 services (44.4%)


In [12]:

pnr_agg = df_pnr.groupby('service_id_pnr').agg(
    total_pax            = ('tcn', 'count'),
    unique_pnrs          = ('pnr_identifier', 'nunique'),
    has_equipment_change = ('equipment_change', lambda x: int((x == 'Y').any())),
    pct_adult            = ('passenger_type', lambda x: (x == 'AD').mean()),
    pct_youth            = ('passenger_type', lambda x: (x == 'YTH').mean()),
    pct_child            = ('passenger_type', lambda x: (x == 'CH').mean()),
    pct_senior           = ('passenger_type', lambda x: (x == 'SNR').mean()),
).reset_index()

pnr_agg['avg_group_size'] = pnr_agg['total_pax'] / pnr_agg['unique_pnrs']


group_ratio = df_pnr.groupby('service_id_pnr')['pnr_identifier'].apply(
    lambda x: (x.map(x.value_counts()) > 1).mean()
).reset_index()
group_ratio.columns = ['service_id_pnr', 'group_pax_ratio']
pnr_agg = pnr_agg.merge(group_ratio, on='service_id_pnr', how='left')


df = df.merge(
    pnr_agg.rename(columns={'service_id_pnr': 'service_id'}),
    on='service_id', how='left'
)
print(f'Shape after PNR join: {df.shape}')
print(f'total_pax null %: {df["total_pax"].isnull().mean():.1%}')

Shape after PNR join: (178384, 191)
total_pax null %: 39.1%


## Feature Engineering

###  Delay Features 


In [13]:

df['arrival_delay_clipped'] = df['arrival_delay_minute'].clip(lower=0)


df['arrived_early'] = (df['arrival_delay_minute'] < 0).astype(int)
df['early_minutes'] = df['arrival_delay_minute'].clip(upper=0).abs()


df['delay_gt5']  = (df['arrival_delay_clipped'] >= 5).astype(int)
df['delay_gt15'] = (df['arrival_delay_clipped'] >= 15).astype(int)
df['delay_gt30'] = (df['arrival_delay_clipped'] >= 30).astype(int)
df['delay_gt60'] = (df['arrival_delay_clipped'] >= 60).astype(int)


df['delay_bucket'] = pd.cut(
    df['arrival_delay_minute'],
    bins=[-9999, -1, 0, 5, 15, 30, 60, 9999],
    labels=['Early', 'On time', '1-5 min', '6-15 min', '16-30 min', '31-60 min', '>60 min']
)

print(df[['arrived_early','delay_gt5','delay_gt15','delay_gt30','delay_gt60']].mean().round(3))
print(df['delay_bucket'].value_counts().sort_index())

arrived_early    0.193
delay_gt5        0.372
delay_gt15       0.229
delay_gt30       0.132
delay_gt60       0.055
dtype: float64
delay_bucket
Early        34374
On time      35907
1-5 min      46769
6-15 min     21679
16-30 min    17101
31-60 min    13116
>60 min       9438
Name: count, dtype: int64


### Delay Cause Encoding

In [14]:
df['delay_is_internal'] = (df['delay_cause_responsibility'] == 'Internal').astype(int)
df['delay_is_external'] = (df['delay_cause_responsibility'] == 'External').astype(int)

print(df[['delay_is_internal','delay_is_external']].mean().round(3))
print(df['delay_cause_responsibility'].value_counts(dropna=False))

delay_is_internal    0.142
delay_is_external    0.463
dtype: float64
delay_cause_responsibility
External    82556
NaN         68932
Internal    25397
n.a.         1499
Name: count, dtype: int64


In [15]:
print(df['delay_cause_category'].value_counts(dropna=False))

delay_cause_category
NaN                                                                 68932
I6 - Traffic Regulation                                             57827
I4 - Infrastructure Fault                                            8865
M1 - Reliability (Sets malfunctions/defects)                         7905
S1 - Passengers                                                      5703
I5 - Trains related incidents                                        5016
S2 - Congestion                                                      3982
I2 - People                                                          3399
O1 - Driving                                                         2005
I3 - Infra Works                                                     1939
Z1 - Weather related - Safety measures / Speed Restrictions          1830
I1 - Animals                                                         1530
M3 - Lack of Available Trainsets                                     1391
M2 - Late release

In [16]:

cause_group_map = {
    # Infrastructure 
    'I4 - Infrastructure Fault':                                        'Infrastructure',
    'I3 - Infra Works':                                                 'Infrastructure',
    'G1 -Facilities issues (Lift, BMI Fence Platform 1&2\u2026)':      'Infrastructure',

    # Traffic & regulation 
    'I6 - Traffic Regulation':                                          'Traffic_Regulation',
    'A4 - Lost of Pathway as train running late with lost of priority':  'Traffic_Regulation',

    # Weather & safety 
    'Z1 - Weather related - Safety measures / Speed Restrictions':      'Weather_Safety',
    'Z3 - Safety Measures from Authorities':                            'Weather_Safety',
    'Y3 - Emergency Services Intervention':                             'Weather_Safety',
    'Y1 - Bomb alert':                                                  'Weather_Safety',
    'Z2 - Act of vandalism':                                            'Weather_Safety',
    'I7 - Strike - Infrastructure Manager':                             'Weather_Safety',

    # Passengers & congestion
    'S1 - Passengers':                                                  'Passenger_Related',
    'S2 - Congestion':                                                  'Passenger_Related',
    'I2 - People':                                                      'Passenger_Related',
    'I1 - Animals':                                                     'Passenger_Related',
    'I5 - Trains related incidents':                                    'Passenger_Related',

    # Eurostar rolling stock 
    'M1 - Reliability (Sets malfunctions/defects)':                     'Rolling_Stock',
    'M2 - Late release from the depot':                                 'Rolling_Stock',
    'M3 - Lack of Available Trainsets':                                 'Rolling_Stock',
    'M4 - Cleaning':                                                    'Rolling_Stock',

    # Eurostar operations 
    'O1 - Driving':                                                     'Eurostar_Operations',
    'O2 - Train Manager':                                               'Eurostar_Operations',
    'O3 - Planning Rostering Eurostar':                                 'Eurostar_Operations',
    'O4 - OCC (Communication issue, Error...)':                         'Eurostar_Operations',
    'A9 - Additional Operational Stop':                                 'Eurostar_Operations',
    'S4 - Eurostar Terminal Staff & Platform duties':                   'Eurostar_Operations',
    'K1 - Catering':                                                    'Eurostar_Operations',

    # Unknown
    'Unmapped':                                                         'Unknown',
}

df['delay_cause_group'] = df['delay_cause_category'].map(cause_group_map).fillna('No_Delay')

print(df['delay_cause_group'].value_counts())
print(f'\nNull %: {df["delay_cause_group"].isnull().mean():.1%}')

delay_cause_group
No_Delay               68932
Traffic_Regulation     58418
Passenger_Related      19630
Infrastructure         10924
Rolling_Stock          10456
Eurostar_Operations     6071
Weather_Safety          3860
Unknown                   93
Name: count, dtype: int64

Null %: 0.0%


In [17]:
cause_dummies = pd.get_dummies(df['delay_cause_group'], prefix='cause').astype(int)
df = pd.concat([df, cause_dummies], axis=1)

print(df[[c for c in df.columns if c.startswith('cause_')]].sum())

cause_Eurostar_Operations     6071
cause_Infrastructure         10924
cause_No_Delay               68932
cause_Passenger_Related      19630
cause_Rolling_Stock          10456
cause_Traffic_Regulation     58418
cause_Unknown                   93
cause_Weather_Safety          3860
dtype: int64


In [18]:


individual_cause_map = {
    # Infrastructure (significant)
    'I4 - Infrastructure Fault':                                        'I4_InfraFault',
    'I3 - Infra Works':                                                 'I3_InfraWorks',
    # Traffic_Regulation (significant)
    'I6 - Traffic Regulation':                                          'I6_TrafficReg',
    'A4 - Lost of Pathway as train running late with lost of priority':  'A4_LostPathway',
    # Passenger_Related (significant)
    'S1 - Passengers':                                                  'S1_Passengers',
    'S2 - Congestion':                                                  'S2_Congestion',
    'I2 - People':                                                      'I2_People',
    'I1 - Animals':                                                     'I1_Animals',
    'I5 - Trains related incidents':                                    'I5_TrainsIncident',
    # Rolling_Stock (significant)
    'M1 - Reliability (Sets malfunctions/defects)':                     'M1_Reliability',
    'M2 - Late release from the depot':                                 'M2_DepotRelease',
    'M3 - Lack of Available Trainsets':                                 'M3_TrainsetShortage',
    # Eurostar_Operations (significant)
    'O1 - Driving':                                                     'O1_Driving',
    'O2 - Train Manager':                                               'O2_TrainManager',
    'O3 - Planning Rostering Eurostar':                                 'O3_Planning',
    'O4 - OCC (Communication issue, Error...)':                         'O4_OCC',
    'K1 - Catering':                                                    'K1_Catering',
    'S4 - Eurostar Terminal Staff & Platform duties':                   'S4_TerminalStaff',
}

# Too small to stand alone within their (significant) group - pooled together
tiny_in_significant = [
    'G1 -Facilities issues (Lift, BMI Fence Platform 1&2…)',
    'M4 - Cleaning',
    'A9 - Additional Operational Stop',
]

df['delay_cause_select'] = df['delay_cause_category'].map(individual_cause_map)
df.loc[df['delay_cause_category'].isin(tiny_in_significant), 'delay_cause_select'] = 'Other_rare_within_expanded'
df.loc[df['delay_cause_group'] == 'Weather_Safety', 'delay_cause_select'] = 'Weather_Safety'
df.loc[df['delay_cause_category'] == 'Unmapped', 'delay_cause_select'] = 'Unknown'
df['delay_cause_select'] = df['delay_cause_select'].fillna('No_Delay')

print(df['delay_cause_select'].value_counts())
print(f"\nTotal: {df['delay_cause_select'].value_counts().sum()} (should be {len(df)})")

print("\nNPS by selective cause:")
print(df.groupby('delay_cause_select')['question_recommendation_nps_a']
        .agg(['mean', 'count']).round(2).sort_values('mean'))

print("\nResponsibility split by selective cause:")
print(df.groupby('delay_cause_select')['delay_cause_responsibility']
        .value_counts(normalize=True, dropna=False).unstack().round(3))


select_dummies = pd.get_dummies(df['delay_cause_select'], prefix='cause_select').astype(int)
df = pd.concat([df, select_dummies], axis=1)

print(f"\nNew dummy columns added: {[c for c in select_dummies.columns]}")

delay_cause_select
No_Delay                      68932
I6_TrafficReg                 57827
I4_InfraFault                  8865
M1_Reliability                 7905
S1_Passengers                  5703
I5_TrainsIncident              5016
S2_Congestion                  3982
Weather_Safety                 3860
I2_People                      3399
O1_Driving                     2005
I3_InfraWorks                  1939
I1_Animals                     1530
M3_TrainsetShortage            1391
M2_DepotRelease                1126
O4_OCC                         1063
S4_TerminalStaff                999
A4_LostPathway                  591
O2_TrainManager                 573
K1_Catering                     547
Other_rare_within_expanded      526
O3_Planning                     512
Unknown                          93
Name: count, dtype: int64

Total: 178384 (should be 178384)

NPS by selective cause:
                            mean  count
delay_cause_select                     
I2_People               

### Class 

In [19]:
class_map = {
    'Eurostar Standard':    1,
    'Eurostar Plus':        2,
    'Eurostar Premier':     3,
    'Wheelchair Space':     0,
    'Wheelchair Companion': 0
}
df['class_encoded']  = df['metadata_class_of_service'].map(class_map)


print(df['class_encoded'].value_counts().sort_index())


class_encoded
0       599
1    130557
2     37328
3      9900
Name: count, dtype: int64


In [26]:
print(df['metadata_price'].value_counts(dropna=False))

metadata_price
39.0     9243
44.0     7841
49.0     7033
35.0     6283
75.0     4974
         ... 
401.0       1
456.0       1
472.0       1
498.0       1
517.0       1
Name: count, Length: 434, dtype: int64


###  Staff Composite 

In [20]:
staff_cols = [
    'question_staff_manners_staff_were_visible',
    'question_staff_manners_staff_were_friendly_and_welcoming',
    'question_staff_manners_staff_were_helpful'
]
df['staff_composite'] = df[staff_cols].mean(axis=1)

print(df['staff_composite'].describe().round(2))
print(f'Null %: {df["staff_composite"].isnull().mean():.1%}')

count    156770.0
mean         4.13
std          0.85
min           1.0
25%          3.67
50%           4.0
75%           5.0
max           5.0
Name: staff_composite, dtype: Float64
Null %: 12.1%


In [8]:
import numpy as np
import pandas as pd


def cronbach_alpha(dataframe):
    """Calculate Cronbach's alpha using complete rows."""
    complete = dataframe.dropna()
    item_count = complete.shape[1]

    item_variance = complete.var(ddof=1).sum()
    total_variance = complete.sum(axis=1).var(ddof=1)

    return (
        item_count / (item_count - 1)
        * (1 - item_variance / total_variance)
    )


staff_cols = [
    "question_staff_manners_staff_were_visible",
    "question_staff_manners_staff_were_friendly_and_welcoming",
    "question_staff_manners_staff_were_helpful",
]

print("Missingness:")
print(df[staff_cols].isna().mean().round(3))

print("\nDistributions:")
for column in staff_cols:
    print(f"\n{column}")
    print(
        df[column]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
        .round(3)
    )

print("\nSpearman correlations:")
print(df[staff_cols].corr(method="spearman").round(3))

alpha = cronbach_alpha(df[staff_cols])
print(f"\nCronbach's alpha: {alpha:.3f}")

Missingness:
question_staff_manners_staff_were_visible                   0.128
question_staff_manners_staff_were_friendly_and_welcoming    0.151
question_staff_manners_staff_were_helpful                   0.188
dtype: float64

Distributions:

question_staff_manners_staff_were_visible
question_staff_manners_staff_were_visible
1       0.016
2       0.048
3       0.094
4       0.386
5       0.327
<NA>    0.128
Name: proportion, dtype: Float64

question_staff_manners_staff_were_friendly_and_welcoming
question_staff_manners_staff_were_friendly_and_welcoming
1       0.013
2       0.026
3       0.105
4       0.326
5       0.379
<NA>    0.151
Name: proportion, dtype: Float64

question_staff_manners_staff_were_helpful
question_staff_manners_staff_were_helpful
1       0.015
2       0.025
3       0.119
4       0.304
5       0.348
<NA>    0.188
Name: proportion, dtype: Float64

Spearman correlations:
                                                    question_staff_manners_staff_were_visible  \
q

### Cleaning Score 

In [21]:

df['clean_below_90'] = pd.NA
df.loc[df['last_clean_score'].notna(), 'clean_below_90'] = (
    df.loc[df['last_clean_score'].notna(), 'last_clean_score'] < 90
).astype(int)

# 3-band categorical
df['clean_band'] = pd.cut(
    df['last_clean_score'],
    bins=[0, 88, 93, 100],
    labels=['Poor', 'Acceptable', 'Good']
)

print(df['clean_below_90'].value_counts(dropna=False))
print(df['clean_band'].value_counts(dropna=False))

clean_below_90
0       91998
<NA>    62164
1       24222
Name: count, dtype: int64
clean_band
Good          68167
NaN           62164
Acceptable    35223
Poor          12830
Name: count, dtype: int64


### Seasonality & Time Features

In [22]:

df['departure_date'] = pd.to_datetime(df['metadata_departure_date'], errors='coerce')

print(df['departure_date'].dt.month.value_counts().sort_index())
print(df['metadata_departure_hour'].head(10))

departure_date
1     20489
2     17699
3     20311
4     20287
5     19249
6     12740
7     12918
8     14440
9     12656
10     9632
11     9042
12     8921
Name: count, dtype: int64
0    20:26
1    18:04
2    13:31
3    11:53
4    09:03
5    14:40
6    14:31
7    09:01
8    12:40
9    10:07
Name: metadata_departure_hour, dtype: str


In [23]:
# Date features
df['departure_month']   = df['departure_date'].dt.month
df['departure_weekday'] = df['departure_date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend']        = (df['departure_weekday'] >= 5).astype(int)

df['departure_season'] = pd.cut(
    df['departure_month'],
    bins=[0, 3, 6, 9, 12],
    labels=['Winter', 'Spring', 'Summer', 'Autumn']
)

# Hour features
df['departure_hour_int'] = df['metadata_departure_hour'].str[:2].astype(int)

# Peak hours: 07-09 and 16-19
df['is_peak_hour'] = (
    ((df['departure_hour_int'] >= 7)  & (df['departure_hour_int'] <= 9)) |
    ((df['departure_hour_int'] >= 16) & (df['departure_hour_int'] <= 19))
).astype(int)

# Time of day bands
df['time_of_day'] = pd.cut(
    df['departure_hour_int'],
    bins=[-1, 6, 12, 17, 21, 24],
    labels=['Early morning', 'Morning', 'Afternoon', 'Evening', 'Night']
)

print(df['departure_season'].value_counts().sort_index())
print(df['is_weekend'].value_counts())
print(df['is_peak_hour'].value_counts())
print(df['time_of_day'].value_counts())

departure_season
Winter    58499
Spring    52276
Summer    40014
Autumn    27595
Name: count, dtype: int64
is_weekend
0    123730
1     54654
Name: count, dtype: int64
is_peak_hour
0    93819
1    84565
Name: count, dtype: int64
time_of_day
Morning          86037
Afternoon        59794
Evening          26272
Early morning     6226
Night               55
Name: count, dtype: int64


### Customer Profiling

In [24]:
profile_cols = ['question_age', 'question_residence', 'question_main_trip_purpose',
                'question_leisure_reason', 'metadata_trips_l12_months',
                'question_language', 'question_connecting_journey',
                'question_future_consideration']

for col in profile_cols:
    print(f'\n{col}:')
    print(df[col].value_counts(dropna=False))


question_age:
question_age
Between 50 - 64      63052
Between 35 - 49      42484
Between 65 - 74      30685
Between 26 - 34      20197
Over 74 years old    11272
Between 18 - 25      10694
Name: count, dtype: int64

question_residence:
question_residence
United Kingdom                     51802
France                             43608
Belgium                            24760
The Netherlands                    20580
Another country (please select)    15233
Germany                            12253
USA                                10148
Name: count, dtype: int64

question_main_trip_purpose:
question_main_trip_purpose
Leisure                          106831
Visiting friends or relatives     42291
Business                          27806
Regular commute                    1456
Name: count, dtype: int64

question_leisure_reason:
question_leisure_reason
NaN                                                              71553
Travel for a short break                                         683

In [25]:
# Age encoding 
age_map = {
    'Between 18 - 25':   1,
    'Between 26 - 34':   2,
    'Between 35 - 49':   3,
    'Between 50 - 64':   4,
    'Between 65 - 74':   5,
    'Over 74 years old': 6
}
df['age_encoded'] = df['question_age'].map(age_map)


residence_map = {
    'United Kingdom':                     1,
    'France':                             2,
    'Belgium':                            3,
    'The Netherlands':                    4,
    'Germany':                            5,
    'USA':                                6,
    'Another country (please select)':    7
}
df['residence_encoded'] = df['question_residence'].map(residence_map)


df['is_business']  = (df['question_main_trip_purpose'] == 'Business').astype(int)
df['is_leisure']   = (df['question_main_trip_purpose'] == 'Leisure').astype(int)
df['is_commuter']  = (df['question_main_trip_purpose'] == 'Regular commute').astype(int)
df['is_vfr']       = (df['question_main_trip_purpose'] == 'Visiting friends or relatives').astype(int)


df['trips_l12m'] = pd.to_numeric(df['metadata_trips_l12_months'], errors='coerce')
df['trips_l12m'] = df['trips_l12m'].clip(upper=20)
df['is_frequent_traveller'] = (df['trips_l12m'] >= 4).astype(int)


language_map = {'English': 1, 'French': 2, 'Dutch': 3, 'German': 4}
df['language_encoded'] = df['question_language'].map(language_map)


df['has_connection'] = (df['question_connecting_journey'] == 'Yes').astype(int)


future_map = {
    'I would not consider Eurostar':                                            0,
    'I would consider Eurostar along with other companies / modes of transport': 1,
    'Eurostar would be my preferred choice':                                    2,
    'Eurostar is the only company I would consider':                            3
}
df['future_loyalty'] = df['question_future_consideration'].map(future_map)


new_profile_cols = ['age_encoded', 'residence_encoded', 'is_business', 'is_leisure',
                    'is_commuter', 'is_vfr', 'trips_l12m', 'is_frequent_traveller',
                    'language_encoded', 'has_connection', 'future_loyalty']

print(df[new_profile_cols].isnull().mean().round(3))
print(df[new_profile_cols].describe().round(2))

age_encoded              0.0
residence_encoded        0.0
is_business              0.0
is_leisure               0.0
is_commuter              0.0
is_vfr                   0.0
trips_l12m               0.0
is_frequent_traveller    0.0
language_encoded         0.0
has_connection           0.0
future_loyalty           0.0
dtype: float64
       age_encoded  residence_encoded  is_business  is_leisure  is_commuter  \
count    178384.00          178384.00    178384.00   178384.00    178384.00   
mean          3.65               2.94         0.16        0.60         0.01   
std           1.24               1.92         0.36        0.49         0.09   
min           1.00               1.00         0.00        0.00         0.00   
25%           3.00               1.00         0.00        0.00         0.00   
50%           4.00               2.00         0.00        1.00         0.00   
75%           4.00               4.00         0.00        1.00         0.00   
max           6.00               7

### Compensation

In [27]:
print(df['compensation_liability_evouchers'].value_counts(dropna=False))
print(df['compensation_liability_cash'].value_counts(dropna=False))

compensation_liability_evouchers
0.00    168946
0.30      4897
0.75      2464
0.60      2077
Name: count, dtype: int64
compensation_liability_cash
0.00    168946
0.25      4897
0.50      4541
Name: count, dtype: int64


In [28]:
print(df['metadata_disrup'].value_counts(dropna=False))

metadata_disrup
         141310
4|4        7652
4          5309
3|3        4634
2|2        3668
5|5        3609
1|1        2811
3          2434
5          2358
2          1854
99|99      1247
1          1084
99          414
Name: count, dtype: int64


In [29]:

df['had_disruption'] = (df['metadata_disrup'].astype(str).str.strip() != '').astype(int)


df['disrup_type'] = df['metadata_disrup'].astype(str).str.split('|').str[0].str.strip()
df['disrup_type'] = df['disrup_type'].replace({'nan': 'none', '': 'none'})

disrup_map = {'none': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '99': 6}
df['disrup_type_encoded'] = df['disrup_type'].map(disrup_map).fillna(0).astype(int)

print(df['had_disruption'].value_counts())
print(df['disrup_type_encoded'].value_counts().sort_index())

had_disruption
0    141310
1     37074
Name: count, dtype: int64
disrup_type_encoded
0    141310
1      3895
2      5522
3      7068
4     12961
5      5967
6      1661
Name: count, dtype: int64


### Reasons for Choosing

In [30]:
print(df['question_reasons_for_choosing'].value_counts().head(20))
print(f'\nTotal unique values: {df["question_reasons_for_choosing"].nunique()}')

question_reasons_for_choosing
["It was the quickest option"]                                                                                                                                    32680
["I prefer taking the train"]                                                                                                                                     32662
["Other (please specify)"]                                                                                                                                        12210
["It was the most environmentally-friendly option"]                                                                                                               10880
["It was the most environmentally-friendly option","I prefer taking the train"]                                                                                   10714
["It was the quickest option","It was the most environmentally-friendly option","I prefer taking the train"]                      

In [31]:
all_reasons = set()
for val in df['question_reasons_for_choosing'].dropna().unique():
    items = re.findall(r'"([^"]+)"', val)
    all_reasons.update(items)

print(f'Total unique individual reasons: {len(all_reasons)}')
for r in sorted(all_reasons):
    print(f'  - {r}')

Total unique individual reasons: 9
  - I prefer taking the train
  - It was the cheapest option
  - It was the most environmentally-friendly option
  - It was the quickest option
  - Not my choice/Didn't have any other options
  - Other (please specify)
  - Quality of service
  - To avoid additional transfers at my destination
  - To use or earn Club Eurostar points


In [32]:
reasons = {
    'reason_quickest':       'It was the quickest option',
    'reason_prefers_train':  'I prefer taking the train',
    'reason_eco':            'It was the most environmentally-friendly option',
    'reason_no_transfers':   'To avoid additional transfers at my destination',
    'reason_cheapest':       'It was the cheapest option',
    'reason_quality':        'Quality of service',
    'reason_no_choice':      "Not my choice/Didn't have any other options",
    'reason_loyalty_points': 'To use or earn Club Eurostar points',
    'reason_other':          'Other (please specify)',
}

for col_name, reason_text in reasons.items():
    df[col_name] = df['question_reasons_for_choosing'].str.contains(
        reason_text, regex=False, na=False
    ).astype(int)

print(df[[c for c in df.columns if c.startswith('reason_')]].sum().sort_values(ascending=False))
print(df[[c for c in df.columns if c.startswith('reason_')]].mean().round(3).sort_values(ascending=False))

reason_prefers_train     86795
reason_quickest          77087
reason_eco               53226
reason_no_transfers      25418
reason_quality           18446
reason_other             18396
reason_cheapest          18333
reason_loyalty_points     5836
reason_no_choice          4903
dtype: int64
reason_prefers_train     0.487
reason_quickest          0.432
reason_eco               0.298
reason_no_transfers      0.142
reason_cheapest          0.103
reason_quality           0.103
reason_other             0.103
reason_loyalty_points    0.033
reason_no_choice         0.027
dtype: float64


###  Route Flag


In [33]:
df['is_channel'] = (df['route_type'] == 'Channel').astype(int)

print(df['is_channel'].value_counts())
print(f'Channel: {df["is_channel"].mean():.1%} of rows')

is_channel
1    116265
0     62119
Name: count, dtype: int64
Channel: 65.2% of rows


## Verification

In [34]:
cause_select_cols = [c for c in df.columns if c.startswith('cause_select_')]

all_new_cols = [
    # Date/time
    'departure_date', 'departure_month', 'departure_weekday', 'is_weekend',
    'departure_season', 'departure_hour_int', 'is_peak_hour', 'time_of_day',
    # Customer profiling
    'age_encoded', 'residence_encoded', 'is_business', 'is_leisure',
    'is_commuter', 'is_vfr', 'trips_l12m', 'is_frequent_traveller',
    'language_encoded', 'has_connection', 'future_loyalty',
    # Compensation
    'compensation_evoucher', 'compensation_cash',
    'received_compensation',
    # Disruption
    'had_disruption', 'disrup_type', 'disrup_type_encoded',
    # Delay features
    'arrival_delay_clipped', 'arrived_early', 'early_minutes',
    'delay_gt5', 'delay_gt15', 'delay_gt30', 'delay_gt60', 'delay_bucket',
    'delay_is_internal', 'delay_is_external',
    # Delay cause groups
    'delay_cause_group',
    # Cause dummies (group-level)
    'cause_Eurostar_Operations', 'cause_Infrastructure', 'cause_No_Delay',
    'cause_Passenger_Related', 'cause_Rolling_Stock',
    'cause_Traffic_Regulation', 'cause_Unknown', 'cause_Weather_Safety',
    # Delay cause — granular (selective expansion)
    'delay_cause_select',
    # Price/class
    'class_encoded', 'price_quartile',
    # Staff
    'staff_composite',
    # Cleaning
    'clean_below_90', 'clean_band',
    # Reasons for choosing
    'reason_quickest', 'reason_prefers_train', 'reason_eco',
    'reason_no_transfers', 'reason_cheapest', 'reason_quality',
    'reason_no_choice', 'reason_loyalty_points', 'reason_other',
    # Route flag
    'is_channel',
] + cause_select_cols

present_new = [c for c in all_new_cols if c in df.columns]
missing_new = [c for c in all_new_cols if c not in df.columns]
print(f'Total new columns to save: {len(present_new)}')
print(f'Missing: {missing_new}')
print(f'Shape: {df.shape}')
print(f'\nNull % on new columns:')
print(df[present_new].isnull().mean().round(3))

Total new columns to save: 78
Missing: ['compensation_evoucher', 'compensation_cash', 'received_compensation', 'price_quartile']
Shape: (178384, 269)

Null % on new columns:
departure_date                   0.0
departure_month                  0.0
departure_weekday                0.0
is_weekend                       0.0
departure_season                 0.0
                                ... 
cause_select_S1_Passengers       0.0
cause_select_S2_Congestion       0.0
cause_select_S4_TerminalStaff    0.0
cause_select_Unknown             0.0
cause_select_Weather_Safety      0.0
Length: 78, dtype: float64


In [35]:
print(f'Total columns: {len(df.columns)}')
for i, col in enumerate(df.columns):
    null_pct = df[col].isnull().mean()
    dtype = df[col].dtype
    print(f'{i+1:3}. {col} [{dtype}] {null_pct:.1%} null')

Total columns: 269
  1. metadata_booking_date [str] 0.0% null
  2. metadata_class_of_service [str] 0.0% null
  3. metadata_class_of_service_code [str] 58.0% null
  4. metadata_coach_number [str] 0.0% null
  5. metadata_compensation_flag [str] 0.0% null
  6. metadata_currency [str] 0.0% null
  7. metadata_delay_code [str] 59.8% null
  8. metadata_delay_at_arrival [str] 3.8% null
  9. metadata_departure_date [str] 0.0% null
 10. metadata_departure_hour [str] 0.0% null
 11. metadata_destination_station [str] 0.0% null
 12. metadata_destination_station_code [str] 58.0% null
 13. metadata_disrup [str] 0.0% null
 14. metadata_origin_station [str] 0.0% null
 15. metadata_origin_station_code [str] 58.0% null
 16. metadata_price [float64] 0.0% null
 17. metadata_q_total_duration [int64] 0.0% null
 18. metadata_route [str] 0.0% null
 19. metadata_route_code [str] 58.0% null
 20. metadata_seat_number [str] 0.0% null
 21. metadata_segment [str] 46.6% null
 22. metadata_segment_code [str] 78.2% nul

In [36]:
pm_cols = ['pm_days_since_catering', 'pm_days_since_toilet', 'pm_days_since_climate',
           'pm_days_since_interior', 'pm_days_since_comms']
cm_cols = ['cm_open_climate', 'cm_open_wifi', 'cm_open_comms', 'cm_open_interior',
           'cm_open_catering', 'cm_open_toilet', 'cm_open_cleaning']

print('PM missing:', [c for c in pm_cols if c not in df.columns])
print('CM missing:', [c for c in cm_cols if c not in df.columns])

PM missing: []
CM missing: []


In [37]:

expected_cause_select = [
    'cause_select_A4_LostPathway', 'cause_select_I1_Animals', 'cause_select_I2_People',
    'cause_select_I3_InfraWorks', 'cause_select_I4_InfraFault', 'cause_select_I5_TrainsIncident',
    'cause_select_I6_TrafficReg', 'cause_select_K1_Catering', 'cause_select_M1_Reliability',
    'cause_select_M2_DepotRelease', 'cause_select_M3_TrainsetShortage', 'cause_select_No_Delay',
    'cause_select_O1_Driving', 'cause_select_O2_TrainManager', 'cause_select_O3_Planning',
    'cause_select_O4_OCC', 'cause_select_Other_rare_within_expanded', 'cause_select_S1_Passengers',
    'cause_select_S2_Congestion', 'cause_select_S4_TerminalStaff', 'cause_select_Unknown',
    'cause_select_Weather_Safety',
]
select_dummy_cols = [c for c in df.columns if c.startswith('cause_select_')]

print(' cause_select presence ---')
print('delay_cause_select present:', 'delay_cause_select' in df.columns)
print('Missing:', sorted(set(expected_cause_select) - set(select_dummy_cols)))
print('Unexpected:', sorted(set(select_dummy_cols) - set(expected_cause_select)))

print('\n--- mutual exclusivity ---')
row_sums = df[select_dummy_cols].sum(axis=1)
print(f"Rows summing to exactly 1: {(row_sums == 1).sum():,} of {len(df):,}")
if (row_sums != 1).sum() > 0:
    print(row_sums.value_counts())

print('\n--- duplicate columns (whole df) ---')
dupes = df.columns[df.columns.duplicated()].tolist()
print(dupes if dupes else 'none')

print('\n--- PM / CM presence ---')
pm_cols = ['pm_days_since_catering', 'pm_days_since_toilet', 'pm_days_since_climate',
           'pm_days_since_interior', 'pm_days_since_comms']
cm_cols = ['cm_open_climate', 'cm_open_wifi', 'cm_open_comms', 'cm_open_interior',
           'cm_open_catering', 'cm_open_toilet', 'cm_open_cleaning']
print('PM missing:', [c for c in pm_cols if c not in df.columns])
print('CM missing:', [c for c in cm_cols if c not in df.columns])

print('\n--- row count ---')
print(f"{len(df):,} rows (expected 178,384)")

 cause_select presence ---
delay_cause_select present: True
Missing: []
Unexpected: []

--- mutual exclusivity ---
Rows summing to exactly 1: 178,384 of 178,384

--- duplicate columns (whole df) ---
none

--- PM / CM presence ---
PM missing: []
CM missing: []

--- row count ---
178,384 rows (expected 178,384)


In [38]:
df.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Saved to C:\Users\gehan\Downloads\thesis\extracted tables\last update data\combined_df_final.parquet
Final shape: 178,384 rows x 269 columns
